# NLP: Social Media Sentiment and RAG

Notebook for the NLP side of the plan: clean crisis posts, classify sentiment/emotion, infer coarse locations, build a small retrieval index, and generate grounded response commentary.

In [ ]:
import hashlib
import re
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

posts = pd.DataFrame([
    {"id": "p1", "text": "Flood water is rising near the central bridge. We are scared and need evacuation help.", "user_location": "Las Vegas, NV", "label": "negative"},
    {"id": "p2", "text": "Shelter reopened at the school. Volunteers brought water and food.", "user_location": "Clark County", "label": "positive"},
    {"id": "p3", "text": "Authorities will inspect damaged buildings this afternoon.", "user_location": "", "label": "neutral"},
    {"id": "p4", "text": "People are trapped after the road collapsed. This is urgent.", "user_location": "North Las Vegas", "label": "negative"},
])
posts

In [ ]:
def clean_text(text):
    text = re.sub(r"http\S+", "", str(text))
    text = re.sub(r"@\w+", "", text)
    text = text.replace("#", "")
    return re.sub(r"\s+", " ", text).strip()

NEGATIVE = {"afraid", "collapsed", "damage", "damaged", "destroyed", "evacuation", "fear", "flood", "help", "scared", "trapped", "urgent"}
POSITIVE = {"aid", "food", "reopened", "safe", "shelter", "support", "volunteers", "water"}

def lexicon_sentiment(text):
    tokens = set(re.findall(r"[a-z']+", clean_text(text).lower()))
    score = len(tokens & POSITIVE) - len(tokens & NEGATIVE)
    if score > 0:
        return "positive"
    if score < 0:
        return "negative"
    return "neutral"

posts["clean_text"] = posts["text"].map(clean_text)
posts["sentiment"] = posts["clean_text"].map(lexicon_sentiment)
posts[["id", "clean_text", "label", "sentiment"]]

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(posts["label"], posts["sentiment"], labels=["positive", "negative", "neutral"], zero_division=0))

## Coarse Location Inference

This mirrors the backend's deterministic mock geocoder. Replace it with GeoNames, Nominatim, or an approved geocoder when you have real data and terms-of-service coverage.

In [ ]:
def stable_point(location):
    if not location:
        return None
    digest = hashlib.sha256(location.encode("utf-8")).digest()
    value = int.from_bytes(digest[:4], "little") % 10000
    return {"lat": 36.0 + (value % 50) / 100.0, "lon": -115.5 + (value % 100) / 100.0}

posts["point"] = posts["user_location"].map(stable_point)
posts[["id", "user_location", "point", "sentiment"]]

## Tiny Local RAG Prototype

In [ ]:
knowledge = [
    {"source": "Flood Response Baseline", "section": "Evacuation", "text": "Prioritize evacuation support, safe access corridors, shelter, water, sanitation, and urgent medical needs."},
    {"source": "Damage Assessment Baseline", "section": "Inspection", "text": "Major or destroyed labels should trigger field validation, structural inspection, and search-and-rescue triage."},
    {"source": "Social Impact Baseline", "section": "Psychosocial", "text": "High fear or negative sentiment can indicate unmet needs, confusion, and psychological stress in affected areas."},
]

def hash_embed(text, size=384):
    vec = np.zeros(size, dtype="float32")
    for token in re.findall(r"[a-z0-9']+", text.lower()):
        digest = hashlib.blake2b(token.encode("utf-8"), digest_size=8).digest()
        idx = int.from_bytes(digest[:4], "little") % size
        vec[idx] += 1 if digest[4] % 2 == 0 else -1
    norm = np.linalg.norm(vec)
    return vec / norm if norm else vec

doc_vectors = np.vstack([hash_embed(item["text"]) for item in knowledge])

def retrieve(query, top_k=2):
    q = hash_embed(query)
    scores = doc_vectors @ q
    order = np.argsort(scores)[::-1][:top_k]
    return [{**knowledge[i], "score": float(scores[i])} for i in order]

hits = retrieve("major flood damage with scared residents needing evacuation", top_k=3)
pd.DataFrame(hits)

In [ ]:
def grounded_commentary(hits):
    citations = [f"{h['source']} - {h['section']}" for h in hits]
    return {
        "commentary": "Prioritize evacuation, route clearance, shelter, water/sanitation, structural inspection, and psychosocial support. Keep claims tied to retrieved guidance and validate CV predictions in the field.",
        "citations": citations,
    }

grounded_commentary(hits)

## Optional Real Datasets

For real NLP experiments, install the project requirements and fetch CrisisLex, Disaster Tweets, or your approved Twitter/X export into `data/nlp/`. Keep raw user identifiers out of committed files.